# **Universal Notebook Environment Setup**

In [1]:
import os
import sys
import wandb

# --- AUTOMATIC ENVIRONMENT SETUP ---
KAGGLE_RUN = os.path.exists('/kaggle/working')

if KAGGLE_RUN:
    print("Running on Kaggle. Setting up paths...")
    # Kaggle Secret for W&B
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

    # Create a symlink so /content/artifacts works on Kaggle
    os.makedirs('/content', exist_ok=True)
    if not os.path.exists('/content/artifacts'):
        # Map Kaggle input artifacts to Colab path
        os.system('ln -s /kaggle/input /content/artifacts')
else:
    print("Running on Google Colab.")
    # Colab Secret for W&B
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
        wandb.login(key=wandb_api_key)
    except Exception as e:
        print("W&B Secret not found, you may need to login manually.")
        wandb.login()

Running on Kaggle. Setting up paths...


# **Fetch Augmented Images and Model From W&B**

In [2]:
import os
import sys
import wandb

# Initialize a single W&B run for environment setup
run = wandb.init(project="pcb-defect-detection", job_type="setup")

print("--- Downloading Dataset Artifact ---")
# 1. Download Dataset
artifact_dataset = run.use_artifact('pcb-augmented-dataset:latest', type='dataset')
dataset_dir = artifact_dataset.download()
print(f"Dataset ready at: {os.path.abspath(dataset_dir)}")

print("\n--- Downloading Core Model Artifact ---")
# 2. Download Core Model Code
artifact_code = run.use_artifact('pcb-core-models:latest', type='model')
model_dir = artifact_code.download()

# Add to sys.path to allow immediate import
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)
print(f"Core model ready at: {model_dir}")

run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: naufalsatya (nsp-deep-learning-projects) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


--- Downloading Dataset Artifact ---


wandb: Downloading large artifact 'pcb-augmented-dataset:latest', 1636.48MB. 5544 files...
wandb:   5544 of 5544 files downloaded.  
Done. 00:00:56.9 (28.7MB/s)


Dataset ready at: /kaggle/working/artifacts/pcb-augmented-dataset:v3

--- Downloading Core Model Artifact ---


wandb:   2 of 2 files downloaded.  


Core model ready at: /kaggle/working/artifacts/pcb-core-models:v3


# **Device Setup & W&B Tracking Initialization**

In [3]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import Dataset, DataLoader

# Initialize the experiment tracking run
run = wandb.init(
    project="pcb-defect-detection",
    name="Model_Architecture_Training",
    notes="Training the MobileViT + LeYOLO architecture and metric logging with validation",
    config = {
        "epochs": 150,
        "batch_size": 16,
        "image_size": 640,
        "loss": "v8DetectionLoss",
        "optimizer": "AdamW|lr=1e-3|weight_decay=1e-4",
        "scheduler": "CosineAnnealingLR|eta_min=1e-6",
        "conf_threshold": 0.05,
        "iou_threshold": 0.45,
        "train/val_split": "80/20",
        "backbone": "mobilevit_xxs(timm,pretrained=True)",
        "multiplier": 1.5,
        "neck_depth": 2,
        "use_sppf": True,
    },
)
config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# **Data Loader From W&B**

In [4]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.5 MB/s eta 0:00:0000:010:01


In [5]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

class PCBDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=config.image_size):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.img_size  = img_size
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Load and preprocess image
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        # Normalize and convert to tensor (CHW format)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Load YOLO format labels
        label_path = os.path.join(self.label_dir, self.img_names[idx].replace('.jpg', '.txt'))
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        labels.append(int(float(parts[0])))  # handles '5.0' format
                        boxes.append([float(x) for x in parts[1:]])

        targets = {
            "boxes":   torch.tensor(boxes,  dtype=torch.float32),
            "labels":  torch.tensor(labels, dtype=torch.int64),
            "raw_img": img  # kept for W&B visualization
        }
        return img_tensor, targets


def collate_fn(batch, device):
    images   = torch.stack([item[0] for item in batch])
    raw_imgs = [item[1]["raw_img"] for item in batch]

    batch_idx_list, cls_list, box_list = [], [], []

    for b_idx, item in enumerate(batch):
        targets   = item[1]
        num_boxes = len(targets["labels"])
        if num_boxes > 0:
            batch_idx_list.append(torch.full((num_boxes,), b_idx, dtype=torch.long))
            cls_list.append(targets["labels"].unsqueeze(1))
            box_list.append(targets["boxes"])

    # Fixed: plain torch.cat — no broken markdown hyperlinks
    if len(batch_idx_list) > 0:
        batch_dict = {
            'batch_idx': torch.cat(batch_idx_list, dim=0).to(device),
            'cls':       torch.cat(cls_list,       dim=0).to(device),
            'bboxes':    torch.cat(box_list,        dim=0).to(device)
        }
    else:
        batch_dict = {
            'batch_idx': torch.empty(0,       dtype=torch.long).to(device),
            'cls':       torch.empty((0, 1),  dtype=torch.long).to(device),
            'bboxes':    torch.empty((0, 4),  dtype=torch.float32).to(device)
        }

    return images, batch_dict, raw_imgs


# ── DATASET & DATALOADER SETUP ─────────────────────────────────────────────────
DATA_DIR     = os.path.join(dataset_dir, 'train')
full_dataset = PCBDataset(
    os.path.join(DATA_DIR, "images"),
    os.path.join(DATA_DIR, "labels")
)

# 80/20 train/val split — fixed seed for reproducibility
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = total_size - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Collate_fn now receives device via lambda — no global dependency
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, device)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,  # order doesn't matter for evaluation
    collate_fn=lambda b: collate_fn(b, device)
)

print(f"Dataset split — Train: {train_size} | Val: {val_size}")

Dataset split — Train: 2217 | Val: 555


# **Tune the LeYOLO + MobileViT Model Architecture**

In [6]:
import importlib.util
import os
import torch
import torch.nn as nn
import timm

def load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module {module_name} from {file_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

model_mobilevit_path = os.path.join(model_dir, "mobilevit_xxs.py")
model_leyolo_path = os.path.join(model_dir, "leyolo_head.py")

mobilevit_xxs = load_module("mobilevit_xxs", model_mobilevit_path)
leyolo_head = load_module("leyolo_head", model_leyolo_path)

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = nn.Conv2d(c1, c_, 1, 1, 0, bias=False)
        self.bn1 = nn.BatchNorm2d(c_)
        self.act1 = nn.SiLU()
        self.cv2 = nn.Conv2d(c_ * 4, c2, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.act2 = nn.SiLU()
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.act1(self.bn1(self.cv1(x)))
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.act2(self.bn2(self.cv2(torch.cat((x, y1, y2, self.m(y2)), 1))))

class ECA(nn.Module):
    def __init__(self, channels: int, k_size: int = 3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2))
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y

class TunedLeNeckBlock(nn.Module):
    def __init__(self, c1, c2, k=3, e=None, stride=1, pw=True, use_attention: bool = True):
        super().__init__()
        c_mid = e if e is not None else c1
        self.residual = c1 == c2 and stride == 1

        layers = []
        if pw and c_mid != c1:
            layers.extend([
                nn.Conv2d(c1, c_mid, kernel_size=1, bias=False),
                nn.BatchNorm2d(c_mid),
                nn.SiLU(),
            ])

        layers.extend([
            nn.Conv2d(c_mid, c_mid, kernel_size=k, stride=stride, padding=k // 2, groups=c_mid, bias=False),
            nn.BatchNorm2d(c_mid),
            nn.SiLU(),
            nn.Conv2d(c_mid, c2, kernel_size=1, bias=False),
            nn.BatchNorm2d(c2),
        ])

        self.layers = nn.Sequential(*layers)
        self.attention = ECA(c2) if use_attention else nn.Identity()

    def forward(self, x):
        out = self.layers(x)
        if self.residual:
            out = out + x
        return self.attention(out)

class TunedLeNeck(nn.Module):
    def __init__(self, width, multiplier=config.multiplier, depth=2):  
        super().__init__()
        self.up = nn.Upsample(scale_factor=2)

        self.p4_up = nn.Sequential(
            TunedLeNeckBlock(c1=width[1]+width[2], c2=int(64*multiplier), e=int(128*multiplier), k=5),
            *[TunedLeNeckBlock(c1=int(64*multiplier), c2=int(64*multiplier), e=int(128*multiplier), k=5)
              for _ in range(int(2*depth))]
        )
        self.p3_up = nn.Sequential(
            TunedLeNeckBlock(c1=int(64*multiplier)+width[0], c2=int(32*multiplier),
                             e=int(64*multiplier)+width[0], k=3, pw=False),
            *[TunedLeNeckBlock(c1=int(32*multiplier), c2=int(32*multiplier), e=int(96*multiplier), k=3)
              for _ in range(int(2*depth))]
        )

        # ✅ mn_conv for downsampling — same as original LeNeck
        self.p3_downsampling = leyolo_head.mn_conv(int(32*multiplier), int(64*multiplier), k=3, s=2, p=1)

        self.p4_down = nn.Sequential(
            TunedLeNeckBlock(c1=int(64*multiplier)*2, c2=int(64*multiplier), e=int(128*multiplier), k=5),
            *[TunedLeNeckBlock(c1=int(64*multiplier), c2=int(64*multiplier), e=int(128*multiplier), k=5)
              for _ in range(int(2*depth))]
        )

        self.p4_downsampling = leyolo_head.mn_conv(int(64*multiplier), int(96*multiplier), k=3, s=2, p=1)

        self.p5_down = nn.Sequential(
            TunedLeNeckBlock(c1=int(96*multiplier)+width[2], c2=int(96*multiplier),
                             e=int(96*multiplier)+width[2], k=5),
            *[TunedLeNeckBlock(c1=int(96*multiplier), c2=int(96*multiplier), e=int(192*multiplier), k=5)
              for _ in range(int(2*depth))]
        )

    def forward(self, x):
        p3, p4, p5 = x
        p4_up   = self.p4_up(torch.cat([self.up(p5), p4], dim=1))
        p3_up   = self.p3_up(torch.cat([self.up(p4_up), p3], dim=1))
        p4_down = self.p4_down(torch.cat([self.p3_downsampling(p3_up), p4_up], dim=1))
        p5_down = self.p5_down(torch.cat([self.p4_downsampling(p4_down), p5], dim=1))
        return p3_up, p4_down, p5_down  # ✅ 3 scales — identical to original

class TunedLeYOLO(nn.Module):
    def __init__(self, num_classes, multiplier=config.multiplier, depth=2, image_size=config.image_size):
        super().__init__()

        # ✅ Pretrained MobileViT-XXS from timm — same architecture as the repo
        # out_indices=(2,3,4) gives [48, 64, 320] channels at strides [8, 16, 32]
        self.net = timm.create_model(
            'mobilevit_xxs',
            pretrained=True,
            features_only=True,
            out_indices=(2, 3, 4)
        )

        width = [48, 64, 320]  # unchanged
        self.sppf = SPPF(width[2], width[2])
        self.fpn  = TunedLeNeck(width, multiplier=multiplier, depth=depth)
        self.head = leyolo_head.LeHead(
            num_classes,
            (int(32*multiplier), int(64*multiplier), int(96*multiplier))
        )

        # Stride calculation — model starts in train mode by default
        img_dummy        = torch.zeros(1, 3, image_size, image_size)
        self.head.stride = torch.tensor(
            [image_size / x.shape[-2] for x in self.forward(img_dummy)]
        )
        self.stride = self.head.stride
        self.head.initialize_biases()

    def forward(self, x):
        features    = self.net(x)       # returns list of 3 tensors
        features[2] = self.sppf(features[2])  # SPPF on P5
        features    = self.fpn(features)
        return self.head(list(features))

# Configure tuned parameters from W&B config (fallback to defaults)
neck_multiplier = getattr(config, "multiplier", 1.5)
neck_depth = getattr(config, "neck_depth", 2)
use_sppf = getattr(config, "use_sppf", True)

model = TunedLeYOLO(
    num_classes=6,
    multiplier=config.multiplier, 
    depth=2,          # ✅ original default
    image_size=config.image_size,
).to(device)

print("Tuned LeYOLO initialized from core architectures.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


model.safetensors:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

Tuned LeYOLO initialized from core architectures.


In [7]:
model = TunedLeYOLO(num_classes=6, multiplier=config.multiplier, depth=2, image_size=config.image_size).to(device)

model.train()
img_dummy = torch.zeros(1, 3, 640, 640).to(device)
with torch.no_grad():
    out = model(img_dummy)

print(f"Scales: {len(out)} | Strides: {model.head.stride}")
for i, o in enumerate(out):
    print(f"  Scale {i}: {o.shape}")

Scales: 3 | Strides: tensor([ 8., 16., 32.])
  Scale 0: torch.Size([1, 70, 80, 80])
  Scale 1: torch.Size([1, 70, 40, 40])
  Scale 2: torch.Size([1, 70, 20, 20])


In [8]:
backbone = timm.create_model('mobilevit_xxs', pretrained=True, features_only=True, out_indices=(2, 3, 4))
dummy = torch.zeros(1, 3, 480, 480)
feats = backbone(dummy)
print([f.shape for f in feats])

[torch.Size([1, 48, 60, 60]), torch.Size([1, 64, 30, 30]), torch.Size([1, 320, 15, 15])]


# **Custom Model Evaluation Function**

In [9]:
import torch
import torchvision.ops as ops
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_model(
    model,
    val_loader,
    device,
    conf_threshold: float = config.conf_threshold,  
    iou_threshold: float = config.iou_threshold,
) -> dict:

    model.eval()

    # Local instance — no global state pollution
    metric = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=True,
        max_detection_thresholds=[1, 10, 50]  
    )

    with torch.no_grad():
        for images, target_dict, _ in val_loader:
            images = images.to(device)

            # Dynamic sizing — works for any resolution
            _, _, H, W = images.shape

            decoded_preds = model(images)  # [B, 4+NC, Anchors]

            preds_list, targets_list = [], []

            for b in range(images.shape[0]):
                # ── GROUND TRUTH ──────────────────────────────────────────
                gt_mask = target_dict['batch_idx'] == b
                gt_boxes_norm = target_dict['bboxes'][gt_mask]  # normalized cxcywh
                # Explicit dtype — avoids silent mismatch in torchmetrics
                gt_labels = target_dict['cls'][gt_mask].squeeze(-1).to(torch.int64)

                if len(gt_boxes_norm) > 0:
                    x_c, y_c, bw, bh = gt_boxes_norm.unbind(1)
                    gt_boxes_xyxy = torch.stack([
                        (x_c - bw / 2) * W, (y_c - bh / 2) * H,
                        (x_c + bw / 2) * W, (y_c + bh / 2) * H,
                    ], dim=1).to(device)
                else:
                    gt_boxes_xyxy = torch.empty((0, 4), device=device)

                targets_list.append({
                    "boxes":  gt_boxes_xyxy,
                    "labels": gt_labels.to(device),
                })

                # ── PREDICTIONS + NMS ──────────────────────────────────────
                preds       = decoded_preds[b]
                pred_boxes  = preds[:4, :].T
                pred_scores = preds[4:, :].T
                
                # ✅ YOLOv8 head already applies sigmoid in eval mode! Do not double sigmoid.
                max_scores, class_indices = pred_scores.max(dim=1)
                conf_mask = max_scores > conf_threshold
                
                f_boxes  = pred_boxes[conf_mask]
                f_scores = max_scores[conf_mask]
                f_labels = class_indices[conf_mask]

                if len(f_boxes) > 0:
                    x_c, y_c, bw, bh = f_boxes.unbind(1)
                    f_boxes_xyxy = torch.stack([
                        (x_c - bw / 2), (y_c - bh / 2),
                        (x_c + bw / 2), (y_c + bh / 2),
                    ], dim=1)

                    keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

                    preds_list.append({
                        "boxes":  f_boxes_xyxy[keep],
                        "scores": f_scores[keep],
                        "labels": f_labels[keep].to(torch.int64),
                    })
                else:
                    preds_list.append({
                        "boxes":  torch.empty((0, 4),                   device=device),
                        "scores": torch.empty((0,),                     device=device),
                        "labels": torch.empty((0,), dtype=torch.int64,  device=device),
                    })

            metric.update(preds_list, targets_list)

    results = metric.compute()

    # Restore training mode before returning
    model.train()
    return results

print("Custom model evaluation initialized.")

Custom model evaluation initialized.


# **Create Visualize Predictions for Validation**

In [10]:
def visualize_predictions(model, viz_batch, device, epoch,
                           conf_threshold=config.conf_threshold,  
                           iou_threshold=config.iou_threshold):

    model.eval()
    drawn_images = []
    with torch.no_grad():
        # Iterate over the pre-collected fixed evaluation batches
        for viz_images, viz_targets, viz_raw in viz_batch:
            viz_images = viz_images.to(device)
            decoded_preds = model(viz_images)

            viz_img = viz_raw[0].copy()
            h, w, _ = viz_img.shape

            # ── GROUND TRUTH (Green) ───────────────────────────────────────────
            gt_mask = viz_targets['batch_idx'] == 0
            for box in viz_targets['bboxes'][gt_mask]:
                x_c, y_c, bw, bh = box.cpu().numpy()
                x1, y1 = int((x_c - bw/2)*w), int((y_c - bh/2)*h)
                x2, y2 = int((x_c + bw/2)*w), int((y_c + bh/2)*h)
                cv2.rectangle(viz_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # ── PREDICTIONS (Red) with NMS ────────────────────────────────────
            preds = decoded_preds[0].cpu()
            pred_boxes  = preds[:4, :].T   # [Anchors, 4] cxcywh
            pred_scores = preds[4:, :].T   # [Anchors, NC]

            # YOLOv8 head already applies sigmoid in eval mode!
            max_scores, class_indices = pred_scores.max(dim=1)
            conf_mask = max_scores > conf_threshold

            f_boxes   = pred_boxes[conf_mask]
            f_scores  = max_scores[conf_mask]
            f_labels  = class_indices[conf_mask]

            if len(f_boxes) > 0:
                x_c, y_c, bw, bh = f_boxes.unbind(1)
                f_boxes_xyxy = torch.stack([
                    (x_c - bw/2), (y_c - bh/2),
                    (x_c + bw/2), (y_c + bh/2)
                ], dim=1)
                keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

                for idx in keep:
                    bx1, by1, bx2, by2 = f_boxes_xyxy[idx].numpy()
                    score  = f_scores[idx].item()
                    cls_id = f_labels[idx].item()  
                    cv2.rectangle(viz_img, (int(bx1), int(by1)), (int(bx2), int(by2)), (255, 0, 0), 2)
                    cv2.putText(
                        viz_img,
                        f"cls{cls_id}: {score:.2f}",
                        (int(bx1), int(by1) - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1
                    )
            
            drawn_images.append(viz_img)

    model.train()
    return drawn_images

print("Visual prediction initialized.")

Visual prediction initialized.


# **Overfit Loop & W&B Training Tracking**

In [ ]:
import os
import torch
import torchvision.ops as ops
from torch.optim.lr_scheduler import CosineAnnealingLR
from ultralytics.utils.loss import v8DetectionLoss

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── 1. DUMMY WRAPPER ────────────────────────────────────────────────────────
class DummyModelConfig:
    def __init__(self, full_model, target_device):
        self._full_model = full_model
        self.device      = target_device
        class Args:
            box, cls, dfl, cls_pw = 7.5, 0.5, 1.5, 1.0
        self.args = Args()
        class MockDetectHead:
            def __init__(self, head, target_device):
                self.stride  = head.stride.to(target_device)  # ✅ ensure on correct device
                self.nc      = head.nc
                self.no      = head.no
                self.device  = target_device
                self.reg_max = head.ch
                self.use_dfl = True
        self.model = [MockDetectHead(full_model.head, target_device='cuda')]
    def parameters(self):
        return self._full_model.parameters()

# ── 2. LOSS, OPTIMIZER & SCHEDULER ──────────────────────────────────────────
dummy_config = DummyModelConfig(model, device)
yolo_loss_fn = v8DetectionLoss(dummy_config)

epochs    = config.epochs
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

# ── 3. BEST MODEL TRACKING ──────────────────────────────────────────────────
best_map50 = 0.0
best_epoch = 0

# Run once before the training loop
VIZ_SAMPLES = 4  # one per difficulty level roughly
viz_batch = []
for i, (imgs, targets, raws) in enumerate(val_loader):
    viz_batch.append((imgs, targets, raws))
    if i >= VIZ_SAMPLES:
        break

# stop if no improvement for 5 evaluation intervals (50 epochs)
patience = 5 # with eval every 10 epochs = 50 epochs patience effectively
no_improve = 0

print(f"Launching Training Run ({epochs} epochs)...")

# ── 4. TRAINING LOOP ────────────────────────────────────────────────────────
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, target_dict, raw_imgs) in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad()
    
        # Get raw feature maps from model — list of 3 tensors in training mode
        feats = model(images)  # [(B, 70, 60, 60), (B, 70, 30, 30), (B, 70, 15, 15)]
    
        # ✅ Build the dict v8DetectionLoss 8.4.47 expects
        box_ch = model.head.ch * 4  # 16 * 4 = 64
        nc     = model.head.nc      # 6
    
        boxes_list  = []
        scores_list = []
        for f in feats:
            B, C, H, W = f.shape
            f_flat = f.view(B, C, -1)              # [B, 70, H*W]
            boxes_list.append(f_flat[:, :box_ch])  # [B, 64, H*W]
            scores_list.append(f_flat[:, box_ch:]) # [B, 6,  H*W]
    
        preds = {
            "feats":  feats,                                          # raw list for make_anchors
            "boxes":  torch.cat(boxes_list,  dim=2),                  # [B, 64, total_anchors]
            "scores": torch.cat(scores_list, dim=2),                  # [B, 6,  total_anchors]
        }
    
        loss, loss_items = yolo_loss_fn(preds, target_dict)
        loss = loss.sum()  # ✅ collapse [box, cls, dfl] into scalar before backward
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_epoch_loss = epoch_loss / len(train_loader)
    current_lr     = scheduler.get_last_lr()[0]

    wandb.log({
        "Train_Loss":    avg_epoch_loss,
        "Learning_Rate": current_lr,
        "Epoch":         epoch
    })

    # ── VALIDATION ──────────────────────────────────────────────────────────
    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"\nRunning Evaluation — Epoch [{epoch}/{epochs}]...")

        val_metrics = evaluate_model(
            model, val_loader, device,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )
        # Pass the pre-collected viz_batch instead of val_loader
        viz_imgs = visualize_predictions(
            model, viz_batch, device, epoch,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )

        current_map50 = val_metrics['map_50'].item()

        wandb.log({
            "Val_mAP_50":             current_map50,
            "Val_Recall":             val_metrics['mar_50'].item(),
            "Val_mAP_50_95":          val_metrics['map'].item(),
            # Log multiple images correctly
            "Validation/Predictions": [wandb.Image(img, caption=f"Epoch {epoch} - Sample {i+1}") for i, img in enumerate(viz_imgs)],
            "Epoch":                  epoch
        })

        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_epoch_loss:.4f} "
              f"| mAP@0.5: {current_map50:.4f} "
              f"| Recall: {val_metrics['mar_50'].item():.4f} "
              f"| LR: {current_lr:.6f}")

        if current_map50 > best_map50:
            best_map50   = current_map50
            best_epoch   = epoch
            no_improve   = 0
            torch.save(model.state_dict(), "mobilevit_leyolo_best.pt")
            print(f"  ★ New best model saved — mAP@0.5: {best_map50:.4f} at epoch {best_epoch}")
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve * 10} epochs (patience: {patience * 10})")
            if no_improve >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch}.")
                print(f"Best mAP@0.5: {best_map50:.4f} at epoch {best_epoch}")
                break

# ── 5. SAVE FINAL WEIGHTS ───────────────────────────────────────────────────
torch.save(model.state_dict(), "mobilevit_leyolo_final.pt")
wandb.save("mobilevit_leyolo_best.pt")
wandb.save("mobilevit_leyolo_final.pt")
wandb.finish()

print(f"\nTraining Complete!")
print(f"  Best : mobilevit_leyolo_best.pt  (epoch {best_epoch}, mAP@0.5: {best_map50:.4f})")
print(f"  Final: mobilevit_leyolo_final.pt")

Launching Training Run (150 epochs)...

Running Evaluation — Epoch [0/150]...
Epoch [0/150] | Loss: 236.5941 | mAP@0.5: 0.0243 | Recall: 0.0248 | LR: 0.001000
  ★ New best model saved — mAP@0.5: 0.0243 at epoch 0

Running Evaluation — Epoch [10/150]...
Epoch [10/150] | Loss: 63.8340 | mAP@0.5: 0.6863 | Recall: 0.3669 | LR: 0.000987
  ★ New best model saved — mAP@0.5: 0.6863 at epoch 10

Running Evaluation — Epoch [20/150]...
Epoch [20/150] | Loss: 51.1103 | mAP@0.5: 0.7471 | Recall: 0.4104 | LR: 0.000952
  ★ New best model saved — mAP@0.5: 0.7471 at epoch 20

Running Evaluation — Epoch [30/150]...
Epoch [30/150] | Loss: 44.5018 | mAP@0.5: 0.7512 | Recall: 0.4221 | LR: 0.000898
  ★ New best model saved — mAP@0.5: 0.7512 at epoch 30

Running Evaluation — Epoch [40/150]...
Epoch [40/150] | Loss: 40.7521 | mAP@0.5: 0.6990 | Recall: 0.3697 | LR: 0.000827
  No improvement for 10 epochs (patience: 200)

Running Evaluation — Epoch [50/150]...


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 50 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Epoch [50/150] | Loss: 36.0945 | mAP@0.5: 0.7574 | Recall: 0.4290 | LR: 0.000741
  ★ New best model saved — mAP@0.5: 0.7574 at epoch 50

Running Evaluation — Epoch [60/150]...
Epoch [60/150] | Loss: 32.3313 | mAP@0.5: 0.7452 | Recall: 0.4221 | LR: 0.000645
  No improvement for 10 epochs (patience: 200)

Running Evaluation — Epoch [70/150]...
Epoch [70/150] | Loss: 30.0539 | mAP@0.5: 0.7377 | Recall: 0.4211 | LR: 0.000542
  No improvement for 20 epochs (patience: 200)

Running Evaluation — Epoch [80/150]...
Epoch [80/150] | Loss: 27.5264 | mAP@0.5: 0.7371 | Recall: 0.4251 | LR: 0.000438
  No improvement for 30 epochs (patience: 200)

Running Evaluation — Epoch [90/150]...
Epoch [90/150] | Loss: 25.7376 | mAP@0.5: 0.7246 | Recall: 0.4205 | LR: 0.000336
  No improvement for 40 epochs (patience: 200)

Running Evaluation — Epoch [100/150]...
Epoch [100/150] | Loss: 23.7364 | mAP@0.5: 0.7282 | Recall: 0.4273 | LR: 0.000242
  No improvement for 50 epochs (patience: 200)

Running Evaluation — 

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch [149/150] | Loss: 20.1245 | mAP@0.5: 0.7248 | Recall: 0.4199 | LR: 0.000001
  No improvement for 100 epochs (patience: 200)


Epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇██
Learning_Rate,█████████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
Train_Loss,█▆▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Val_Recall,▁▇██▇███████████
Val_mAP_50,▁▇██▇███████████
Val_mAP_50_95,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,149
Learning_Rate,0.0
Train_Loss,20.12449
Val_Recall,0.41994
Val_mAP_50,0.72476



Training Complete!
  Best : mobilevit_leyolo_best.pt  (epoch 50, mAP@0.5: 0.7574)
  Final: mobilevit_leyolo_final.pt
